# Aplicações Avançadas de Chatbots com NLP

Vamos construir um **chatbot de atendimento** para uma empresa fictícia.

O chatbot deverá conseguir:

1. reconhecer a intenção do usuário;
2. identificar informações importantes na mensagem;
3. manter algumas informações da conversa;
4. consultar dados externos;
5. responder quando tiver confiança suficiente;
6. recusar ou encaminhar solicitações que não consegue compreender.

> **Importante:** usaremos o JSONPlaceholder como uma API pública de testes. Ele fornece recursos genéricos como `users`, `posts` e `todos`. Para fins didáticos, vamos interpretar `users` como clientes e `todos` como uma representação simplificada de pedidos/tarefas. Isso é uma simulação educacional, não um sistema de e-commerce real.

## 1. Do chatbot baseado em intenção para um chatbot orientado a ações

No encontro anterior, uma mensagem como:

> "Quanto custa o notebook?"

podia ser classificada como:

```text
intenção = preco
```

Agora queremos ir além:

```text
Mensagem
   ↓
NLP
   ├── intenção
   └── entidade
          ↓
   Gerenciador de diálogo
          ↓
      ação
          ↓
 API / banco / serviço
          ↓
      resposta
```

A principal mudança é que o chatbot deixa de apenas **classificar uma mensagem** e passa a **usar a classificação para decidir o que fazer**.

In [ ]:
import re
import requests
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

## 2. Conhecendo a API que será integrada

O JSONPlaceholder é uma API REST pública criada para testes, protótipos, exemplos e tutoriais. Ela não exige cadastro nem chave de API.

Entre os recursos disponíveis estão:

- `/users`
- `/posts`
- `/comments`
- `/albums`
- `/photos`
- `/todos`

Nesta aula utilizaremos principalmente:

```text
GET /users/{id}
GET /todos?userId={id}
GET /todos/{id}
```

Fonte: [JSONPlaceholder](https://jsonplaceholder.typicode.com/)

### Atenção

Os dados do JSONPlaceholder são fictícios. Além disso, operações de escrita são simuladas pela API e não devem ser tratadas como persistência real.

In [ ]:
API_BASE = "https://jsonplaceholder.typicode.com"

def get_api(path, params=None):
    """Realiza uma chamada GET e retorna o JSON da API."""
    url = f"{API_BASE}{path}"
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()

In [ ]:
# Primeira chamada: consultar um usuário
cliente = get_api("/users/1")
cliente

### O que aconteceu?

O chatbot agora consegue consultar um sistema externo.

A resposta da API contém informações estruturadas. Por exemplo:

```text
id
name
username
email
address
phone
company
```

Isso é diferente de simplesmente armazenar uma resposta pronta no código.

**Chatbot avançado = NLP + lógica de diálogo + serviços externos.**

## 3. Criando o conjunto de intenções

Vamos utilizar um conjunto pequeno de exemplos para manter o foco na arquitetura.

As intenções serão:

| Intenção | Objetivo |
|---|---|
| `saudacao` | iniciar uma conversa |
| `preco` | consultar preço de um produto |
| `estoque` | consultar disponibilidade |
| `pedido` | consultar pedidos/tarefas |
| `cliente` | consultar informações do cliente |
| `ajuda` | explicar o que o chatbot consegue fazer |

> Os dados de preço e estoque continuarão locais, enquanto clientes e pedidos serão obtidos da API. Assim conseguimos observar duas fontes diferentes de informação.

> Os exemplos de treinamento estão separados em `datasets/dados_treinamento.csv`, na pasta deste encontro.


In [ ]:
# O dataset de treinamento fica em datasets/dados_treinamento.csv.\n",
# Para usar o dataset menor, descomente a linha abaixo e comente a linha seguinte.\n",
#df = pd.read_csv("datasets/dados_treinamento_menor.csv")
df = pd.read_csv("datasets/dados_treinamento.csv")

In [ ]:
df["intencao"].value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["texto"],
    df["intencao"],
    test_size=0.2,
    random_state=42,
    stratify=df["intencao"]
)

print("Treinamento:", len(X_train))
print("Teste:", len(X_test))

## 4. Treinando o classificador

Assim como no encontro anterior, vamos utilizar aprendizado supervisionado.

A diferença é que agora o classificador será apenas uma parte do sistema.

```text
Classificador
      ↓
intenção
      ↓
ação
```

Ele não será responsável sozinho por toda a conversa.

In [ ]:
modelo = Pipeline([
    ("count", CountVectorizer(
        ngram_range=(1, 2),
        lowercase=True
    )),
    ("clf", LogisticRegression(max_iter=1000))
])
modelo.fit(X_train, y_train)

In [ ]:
predicoes = modelo.predict(X_test)

print("Accuracy no conjunto de testes:", accuracy_score(y_test, predicoes))
print()
print(classification_report(y_test, predicoes, zero_division=0))

## 5. Intenção não é entidade

Considere:

> "Quanto custa o notebook?"

Temos:

```text
intenção = preco
entidade = notebook
```

A intenção responde:

> **O que o usuário quer fazer?**

A entidade responde:

> **Sobre qual objeto ou informação ele está falando?**

Outro exemplo:

> "Tem celular disponível?"

```text
intenção = estoque
entidade = celular
```

In [ ]:
PRODUTOS = {
    "notebook": {
        "preco": 3499.90,
        "estoque": 12
    },
    "celular": {
        "preco": 1999.90,
        "estoque": 5
    },
    "tablet": {
        "preco": 1499.90,
        "estoque": 8
    }
}

def extrair_produto(texto):
    texto = texto.lower()

    for produto in PRODUTOS:
        if produto in texto:
            return produto

    return None

In [ ]:
exemplos = [
    "quanto custa o notebook?",
    "tem celular disponível?",
    "qual é o preço do tablet?"
]

for texto in exemplos:
    print(
        texto,
        "-> intenção:", modelo.predict([texto])[0],
        "| produto:", extrair_produto(texto)
    )

## 6. Contexto conversacional

Observe esta conversa:

```text
Usuário: Quanto custa o notebook?
Bot: O notebook custa R$ 3.499,90.

Usuário: E ele está disponível?
```

A segunda frase não informa explicitamente o produto.

O chatbot precisa recuperar uma informação anterior:

```text
produto anterior = notebook
```

Vamos criar uma memória simples.

In [ ]:
contexto = {
    "ultima_intencao": None,
    "produto": None,
    "cliente_id": None,
    "ultimo_pedido_id": None
}

def atualizar_contexto(intencao, produto=None):
    contexto["ultima_intencao"] = intencao

    if produto is not None:
        contexto["produto"] = produto

In [ ]:
texto = "Quanto custa o notebook?"

intencao = modelo.predict([texto])[0]
produto = extrair_produto(texto)

atualizar_contexto(intencao, produto)

contexto

Agora uma nova mensagem:

> "E ele está disponível?"

Não há uma entidade `produto` explícita.

Podemos usar o contexto anterior.

In [ ]:
texto = "E ele está disponível?"

intencao = modelo.predict([texto])[0]
produto = extrair_produto(texto)

if produto is None:
    produto = contexto["produto"]

print("Intenção:", intencao)
print("Produto:", produto)

### Insight

O contexto transforma duas mensagens independentes em uma sequência conversacional:

```text
M1: Quanto custa o notebook?
       ↓
produto = notebook

M2: E ele está disponível?
       ↓
recupera produto do contexto

Resultado:
estoque(notebook)
```

Esse é um exemplo simples de **gerenciamento de diálogo**.

## 7. Consultando informações do sistema externo

Agora vamos conectar a intenção `cliente` a uma API.

A função abaixo consulta os dados de um cliente no JSONPlaceholder.

In [ ]:
def consultar_cliente(cliente_id):
    return get_api(f"/users/{cliente_id}")

In [ ]:
cliente = consultar_cliente(1)

print("Nome:", cliente["name"])
print("E-mail:", cliente["email"])
print("Telefone:", cliente["phone"])

## 8. Consultando pedidos - simulação didática

O JSONPlaceholder não possui um recurso chamado `orders`.

Para demonstrar integração com um sistema externo, vamos utilizar `/todos` como uma **representação didática de pedidos/tarefas** vinculados a um cliente.

A API permite consultar:

```text
GET /todos?userId=1
```

Assim conseguimos mostrar uma situação semelhante a:

```text
Chatbot
   ↓
intenção = pedido
   ↓
cliente_id = 1
   ↓
API externa
   ↓
pedidos do cliente
   ↓
resposta
```

In [ ]:
def consultar_pedidos(cliente_id):
    return get_api("/todos", params={"userId": cliente_id})

pedidos = consultar_pedidos(1)

pd.DataFrame(pedidos).head(10)

In [ ]:
def status_pedido(pedido):
    return "concluído" if pedido["completed"] else "em andamento"

for pedido in pedidos[:5]:
    print(
        f"Pedido {pedido['id']}: "
        f"{status_pedido(pedido)}"
    )

## 9. Integrando NLP + contexto + API

Agora vamos juntar as partes.

A lógica será:

```text
Mensagem
   ↓
Classificação de intenção
   ↓
Extração de entidade
   ↓
Recuperação do contexto
   ↓
Execução da ação
   ↓
Resposta
```

In [ ]:
def responder_preco(produto):
    if produto not in PRODUTOS:
        return "Não encontrei esse produto no catálogo."

    preco = PRODUTOS[produto]["preco"]
    return f"O {produto} custa R$ {preco:,.2f}.".replace(",", "X").replace(".", ",").replace("X", ".")


def responder_estoque(produto):
    if produto not in PRODUTOS:
        return "Não encontrei esse produto no catálogo."

    quantidade = PRODUTOS[produto]["estoque"]

    if quantidade == 0:
        return f"O {produto} está sem estoque."

    return f"Sim! Temos {quantidade} unidade(s) de {produto} em estoque."


def responder_cliente(cliente_id):
    try:
        cliente = consultar_cliente(cliente_id)
        return (
            f"Cadastro encontrado: {cliente['name']}. "
            f"E-mail: {cliente['email']}."
        )
    except requests.RequestException:
        return "Não consegui consultar o cadastro neste momento."


def responder_pedido(cliente_id):
    try:
        pedidos = consultar_pedidos(cliente_id)

        if not pedidos:
            return "Não encontrei pedidos para esse cliente."

        em_andamento = sum(not p["completed"] for p in pedidos)
        concluidos = sum(p["completed"] for p in pedidos)

        return (
            f"Encontrei {len(pedidos)} registro(s): "
            f"{concluidos} concluído(s) e "
            f"{em_andamento} em andamento."
        )
    except requests.RequestException:
        return "Não consegui consultar os pedidos neste momento."

## 10. Confiança do modelo

Um problema importante:

> O classificador sempre pode escolher alguma classe.

Imagine:

```text
"Qual é a capital da França?"
```

Se essa mensagem não pertence ao conjunto de intenções, ainda assim o modelo pode escolher a intenção com maior probabilidade.

Para reduzir esse problema, vamos estabelecer um **limiar de confiança**.

```text
confiança >= 0.60
        ↓
executar intenção

confiança < 0.60
        ↓
fallback
```

O valor `0.60` é apenas um parâmetro didático. Em uma aplicação real, o limiar deve ser avaliado empiricamente.

In [ ]:
def classificar_com_confianca(texto, limiar=0.60):
    probabilidades = modelo.predict_proba([texto])[0]
    indice = probabilidades.argmax()

    intencao = modelo.classes_[indice]
    confianca = probabilidades[indice]

    if confianca < limiar:
        return "fallback", confianca

    return intencao, confianca

In [ ]:
testes = [
    "quanto custa o notebook",
    "tem celular disponível",
    "quero consultar meu pedido",
    "qual é a capital da França?"
]

for texto in testes:
    intencao, confianca = classificar_com_confianca(texto)
    print(f"{texto!r}")
    print(f"  intenção = {intencao}")
    print(f"  confiança = {confianca:.2f}")
    print()

## 11. Construindo o chatbot

Agora podemos criar uma função que representa o fluxo completo.

O chatbot irá:

- classificar a intenção;
- verificar o nível de confiança;
- extrair produto;
- utilizar o contexto;
- consultar a API quando necessário;
- produzir uma resposta;
- atualizar a memória.

In [ ]:
def chatbot(texto):
    texto_lower = texto.lower().strip()

    # 1. Classificação
    intencao, confianca = classificar_com_confianca(texto_lower)

    # 2. Fallback
    if intencao == "fallback":
        return (
            "Não consegui entender sua solicitação com segurança. "
            "Posso ajudar com preço, estoque, pedidos ou cadastro."
        )

    # 3. Entidade
    produto = extrair_produto(texto_lower)

    # 4. Contexto
    if produto is None and intencao in {"preco", "estoque"}:
        produto = contexto["produto"]

    # 5. Ações
    if intencao == "saudacao":
        resposta = "Olá! Como posso ajudar?"

    elif intencao == "ajuda":
        resposta = (
            "Posso consultar preços, verificar estoque, "
            "consultar pedidos e consultar dados de cadastro."
        )

    elif intencao == "preco":
        if produto is None:
            resposta = "Qual produto você deseja consultar?"
        else:
            resposta = responder_preco(produto)

    elif intencao == "estoque":
        if produto is None:
            resposta = "Qual produto você deseja consultar?"
        else:
            resposta = responder_estoque(produto)

    elif intencao == "cliente":
        if contexto["cliente_id"] is None:
            resposta = "Informe seu ID de cliente para eu consultar o cadastro."
        else:
            resposta = responder_cliente(contexto["cliente_id"])

    elif intencao == "pedido":
        if contexto["cliente_id"] is None:
            resposta = "Informe seu ID de cliente para eu consultar seus pedidos."
        else:
            resposta = responder_pedido(contexto["cliente_id"])

    else:
        resposta = "Não tenho uma ação configurada para essa intenção."

    # 6. Atualização do contexto
    atualizar_contexto(intencao, produto)

    return resposta

### 12. Informando o cliente no contexto

Para simplificar a aula, vamos permitir que o usuário informe o ID do cliente.

Exemplo:

```text
"meu cliente é 1"
```

Em uma aplicação real, esse identificador não deveria depender de uma mensagem livre. O sistema poderia obtê-lo a partir de uma sessão autenticada, token ou mecanismo equivalente.

In [ ]:
def atualizar_cliente_contexto(texto):
    padrao = r"(?:cliente|id)\s*(?:é|e|:)?\s*(\d+)"
    match = re.search(padrao, texto.lower())

    if match:
        contexto["cliente_id"] = int(match.group(1))
        return True

    return False

In [ ]:
mensagens = [
    "meu cliente é 1",
    "quero consultar meus pedidos",
    "quero consultar meus dados"
]

for mensagem in mensagens:
    if atualizar_cliente_contexto(mensagem):
        print("Contexto atualizado:", contexto)

    print("Bot:", chatbot(mensagem))
    print()

## 13. Simulando uma conversa completa

Vamos testar uma sequência na qual o chatbot precisa manter contexto.

Exemplo esperado:

```text
Usuário: meu cliente é 1
Bot: ...

Usuário: quero consultar meus pedidos
Bot: ...

Usuário: quanto custa o notebook?
Bot: ...

Usuário: e ele está disponível?
Bot: ...
```

In [ ]:
def conversar(mensagens):
    for mensagem in mensagens:
        atualizar_cliente_contexto(mensagem)

        print("Usuário:", mensagem)
        print("Bot:", chatbot(mensagem))
        print("-" * 60)

conversa = [
    "meu cliente é 1",
    "quero consultar meus pedidos",
    "quanto custa o notebook?",
    "e ele está disponível?"
]

conversar(conversa)

## Referências e documentação

- JSONPlaceholder: https://jsonplaceholder.typicode.com/
- JSONPlaceholder Guide: https://github.com/typicode/jsonplaceholder
- Material da Aula 7 — Aplicações Avançadas de Chatbots com NLP.